# 🦅 បង្វឹកបន្ថែម (Fine-Tuning) ម៉ូដែល Khmer & Math PARSeq OCR រយៈពេល 30 Epochs លើ Kaggle GPU

Notebook នេះត្រូវបានរចនាឡើងយ៉ាងពិសេសសម្រាប់ដំណើរការលើ **Kaggle Notebooks (GPU T4 x2 / P100)** ដោយមានលក្ខណៈពិសេស៖
- 🌐 **បិទ Browser ចោលបាន (Background Execution 12h):** គ្រាន់តែចុច **Save Version -> Save & Run All (Commit)** អ្នកអាចបិទកុំព្យូទ័រចោលបានភ្លាមៗ វានឹងរត់រហូតដល់ចប់ 30 Epochs ដោយស្វ័យប្រវត្តិ!
- 🎯 **Base Checkpoint:** ទាញយកស្វ័យប្រវត្តិពី [`krotreaksmey/TinyOCRkhmer`](https://huggingface.co/krotreaksmey/TinyOCRkhmer) លើ Hugging Face
- 📚 **Dataset:** ទាញយកស្វ័យប្រវត្តិពី [`krotreaksmey/synthetic_khmer_math`](https://huggingface.co/datasets/krotreaksmey/synthetic_khmer_math) (117,541 train + 29,373 val samples)
- ⚡ **LoRA (Rank=16, Alpha=32):** សន្សំសំចៃ VRAM GPU ជាង 70% និងការពារ Catastrophic Forgetting
- 🔄 **Auto-Resume & Save គ្រប់ 1 Epoch:** Save គ្រប់ Checkpoint ទាំងអស់ (Epoch 01-30) + Best Model + Last Model និងអាច **Resume បន្តពី Checkpoint ចាស់បានគ្រប់ពេល**!

> ⚠️ **សំខាន់បំផុតមុនពេលចុច Run លើ Kaggle:**
> នៅផ្ទាំងខាងស្ដាំដៃ (**Notebook settings**):
> 1. **Accelerator:** ជ្រើសរើស **GPU T4 x2** (ឬ GPU P100)
> 2. **Internet:** ត្រូវចុច **បើក (Turn ON)** ដើម្បីឱ្យវាអាចទាញយក Dataset & Model ពី Hugging Face បាន!

## ជំហានទី ១៖ ពិនិត្យ Kaggle GPU & System Environment

In [ ]:
!nvidia-smi

import os
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    print(f"GPU Count: {gpu_count}")
    for i in range(gpu_count):
        print(f"  - GPU {i}: {torch.cuda.get_device_name(i)}")
    torch.backends.cudnn.benchmark = True
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
    print("⚠️ GPU មិនទាន់បើកទេ។ សូមបើក Accelerator -> GPU T4 x2 ក្នុងផ្ទាំងខាងស្ដាំ!")

print(f"Active Device: {device}")

## ជំហានទី ២៖ ដំឡើងបណ្ណាល័យចាំបាច់លើ Kaggle

In [ ]:
!pip install -q datasets timm jiwer huggingface_hub

## ជំហានទី ៣៖ រៀបចំ Folder ផ្ទុកទិន្នន័យលើ Kaggle (`/kaggle/working/`)

In [ ]:
OUTPUT_DIR = "/kaggle/working/khmer_math_ocr_lora_30epoch"
CHECKPOINTS_DIR = os.path.join(OUTPUT_DIR, "epoch_checkpoints")

os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
print(f"📁 Output Directory:        {OUTPUT_DIR}")
print(f"📁 Checkpoints per Epoch:   {CHECKPOINTS_DIR}")

## ជំហានទី ៤៖ ទាញយក Base Model Weights ដោយស្វ័យប្រវត្តិពី Hugging Face (`krotreaksmey/TinyOCRkhmer`)

In [ ]:
from huggingface_hub import hf_hub_download

HF_REPO_ID = "krotreaksmey/TinyOCRkhmer"
MODEL_FILENAME = "best_khmer_parseq_model_30epoch.pt"

print(f"⏳ កំពុងទាញយក Base Weights ពី Hugging Face: {HF_REPO_ID}...")
WEIGHTS_PATH = hf_hub_download(repo_id=HF_REPO_ID, filename=MODEL_FILENAME)
print(f"✅ ទាញយកជោគជ័យ! File Path: {WEIGHTS_PATH} ({os.path.getsize(WEIGHTS_PATH)/1e6:.1f} MB)")

## ជំហានទី ៥៖ ផ្ទុកទិន្នន័យពី Hugging Face (`krotreaksmey/synthetic_khmer_math`)

In [ ]:
from datasets import load_dataset
import matplotlib.pyplot as plt

print("⏳ កំពុងទាញយក Dataset: krotreaksmey/synthetic_khmer_math ...")
dataset = load_dataset("krotreaksmey/synthetic_khmer_math")

train_raw = dataset["train"]
val_raw = dataset["validation"]

print(f"✅ Train Samples:      {len(train_raw):,}")
print(f"✅ Validation Samples: {len(val_raw):,}")

fig, axes = plt.subplots(1, 3, figsize=(15, 3))
for i in range(3):
    sample = train_raw[i]
    axes[i].imshow(sample["image"])
    axes[i].set_title(f"Text: {sample['text'][:25]}...", fontsize=10)
    axes[i].axis("off")
plt.tight_layout()
plt.show()

## ជំហានទី ៦៖ បង្កើត CharTokenizer ពេញលេញ (ខ្មែរ + ឡាតាំង + គណិតវិទ្យា LaTeX)

In [ ]:
from typing import Sequence

class CharTokenizer:
    def __init__(self, chars=None, special_tokens=["<s>", "</s>", "<pad>", "<unk>"]):
        if chars is None:
            kh_charset = "០១២៣៤៥៦៧៨៩កខគឃងចឆជឈញដឋឌឍណតថទធនបផពភមយរលវសហឡអឥឧឳឪឱឫឬឭឮឦឰឯាិីឹឺុូួើឿៀេែៃោៅំះៈ់៉៊៍័៏៌្ ។៕៖ៗ"
            en_charset = "0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ!\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~ "
            latex_charset = "/\\^_{}()+-=<>[]|!@#%&*~;:"
            chars = sorted(list(set(kh_charset + en_charset + latex_charset)))
        self.special_tokens = special_tokens
        self.vocab = tuple(special_tokens[:1]) + tuple(chars) + tuple(special_tokens[1:])
        self.str_to_int = {s: i for i, s in enumerate(self.vocab)}
        self.int_to_str = {i: s for i, s in enumerate(self.vocab)}
        self.bos_id = self.str_to_int["<s>"]
        self.eos_id = self.str_to_int["</s>"]
        self.pad_id = self.str_to_int["<pad>"]
        self.unk_id = self.str_to_int["<unk>"]

    def __len__(self):
        return len(self.vocab)

    def encode(self, text: str, add_special_tokens: bool = True):
        tokens = []
        i = 0
        while i < len(text):
            matched = False
            for spec in self.special_tokens:
                if text.startswith(spec, i):
                    tokens.append(self.str_to_int[spec])
                    i += len(spec)
                    matched = True
                    break
            if not matched:
                char = text[i]
                tokens.append(self.str_to_int.get(char, self.unk_id))
                i += 1
        if add_special_tokens:
            tokens = [self.bos_id] + tokens + [self.eos_id]
        return tokens

    def decode(self, ids: Sequence[int], ignore_special_tokens: bool = True) -> str:
        if ignore_special_tokens:
            return "".join([self.int_to_str[i] for i in ids if self.int_to_str.get(i, '') not in self.special_tokens])
        return "".join([self.int_to_str.get(i, '<unk>') for i in ids])

tokenizer = CharTokenizer()
print(f"✅ Tokenizer Vocab Size: {len(tokenizer)} (ត្រូវគ្នានឹង Model: 185)")

## ជំហានទី ៧៖ រៀបចំ PyTorch Dataset, Data Augmentations & DataLoaders

In [ ]:
from PIL import Image
from torchvision.transforms import v2 as transforms
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

IMG_SIZE = (48, 192)
MAX_LABEL_LEN = 120

train_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomAffine(
        degrees=2,
        translate=(0.02, 0.02),
        scale=(0.97, 1.03),
        fill=255
    ),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToImage(),
    transforms.ToDtype(torch.float32, scale=True),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToImage(),
    transforms.ToDtype(torch.float32, scale=True),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class OCRDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, transform, max_len=MAX_LABEL_LEN):
        self.dataset = hf_dataset
        self.tokenizer = tokenizer
        self.transform = transform
        self.max_len = max_len

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        img = item["image"]
        if img.mode != "RGB":
            img = img.convert("RGB")
        img_tensor = self.transform(img)

        text = str(item.get("text", ""))
        tokens = self.tokenizer.encode(text, add_special_tokens=True)
        if len(tokens) > self.max_len:
            tokens = tokens[:self.max_len - 1] + [self.tokenizer.eos_id]

        tokens_tensor = torch.tensor(tokens, dtype=torch.long)
        return {
            "img": img_tensor,
            "input_tokens": tokens_tensor[:-1],
            "target_tokens": tokens_tensor[1:]
        }

def collate_fn(batch):
    imgs = torch.stack([b["img"] for b in batch])
    input_tokens = pad_sequence([b["input_tokens"] for b in batch], batch_first=True, padding_value=tokenizer.pad_id)
    target_tokens = pad_sequence([b["target_tokens"] for b in batch], batch_first=True, padding_value=tokenizer.pad_id)
    return {"img": imgs, "input_tokens": input_tokens, "target_tokens": target_tokens}

train_dataset = OCRDataset(train_raw, tokenizer, train_transform)
val_dataset = OCRDataset(val_raw, tokenizer, val_transform)

BATCH_SIZE = 128
NUM_WORKERS = 4

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collate_fn, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_fn, pin_memory=True)

print(f"✅ DataLoaders Ready! Batches per Epoch: Train={len(train_loader)}, Val={len(val_loader)}")

## ជំហានទី ៨៖ ស្ថាបត្យកម្មម៉ូដែល PARSeq (VisionTransformer + Autoregressive Decoder)

In [ ]:
import math
import inspect
from dataclasses import dataclass
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from timm.models.vision_transformer import VisionTransformer, PatchEmbed

@dataclass
class ModelConfig:
    img_size: Sequence[int] = (48, 192)
    patch_size: Sequence[int] = (4, 8)
    n_channel: int = 3
    vocab_size: int = 185
    block_size: int = 128
    n_layer: int = 12
    n_head: int = 3
    n_embed: int = 192
    dropout: float = 0.1
    bias: bool = True

class ImageEncoder(VisionTransformer):
    def __init__(self, config: ModelConfig):
        sig = inspect.signature(VisionTransformer.__init__)
        kwargs = {
            "img_size": config.img_size, "patch_size": config.patch_size, "in_chans": config.n_channel,
            "embed_dim": config.n_embed, "depth": config.n_layer, "num_heads": config.n_head,
            "mlp_ratio": 4, "qkv_bias": True, "drop_rate": 0.0, "attn_drop_rate": 0.0, "drop_path_rate": 0.0,
            "embed_layer": PatchEmbed, "num_classes": 0, "global_pool": "", "class_token": False
        }
        valid_kwargs = {k: v for k, v in kwargs.items() if k in sig.parameters}
        super().__init__(**valid_kwargs)
        self.cls_token = None
        num_patches = (config.img_size[0] // config.patch_size[0]) * (config.img_size[1] // config.patch_size[1])
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, config.n_embed))

    def forward(self, x: Tensor) -> Tensor:
        x = self.patch_embed(x)
        x = self.pos_drop(x + self.pos_embed)
        x = self.blocks(x)
        return self.norm(x)

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: Tensor) -> Tensor:
        variance = x.float().pow(2).mean(-1, keepdim=True)
        norm_x = x.float() * torch.rsqrt(variance + self.eps)
        return (self.weight * norm_x).type(x.dtype)

class Linear(nn.Linear):
    def forward(self, x: Tensor) -> Tensor:
        return F.linear(x, self.weight.to(x.dtype), None if self.bias is None else self.bias.to(x.dtype))

class TextDecoder(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config
        self.n_head = 2 * config.n_head
        self.tok_embed = nn.Embedding(config.vocab_size, config.n_embed)
        self.pos_embed = nn.Parameter(torch.Tensor(1, config.block_size, config.n_embed))
        self.dropout = nn.Dropout(config.dropout)
        self.sa_ln = RMSNorm(config.n_embed)
        self.sa_attn = nn.MultiheadAttention(config.n_embed, self.n_head, dropout=config.dropout, batch_first=True)
        self.cross_ln = RMSNorm(config.n_embed)
        self.cross_attn = nn.MultiheadAttention(config.n_embed, self.n_head, dropout=config.dropout, batch_first=True)
        self.ffn_ln = RMSNorm(config.n_embed)
        dim_feedforward = 4 * config.n_embed
        self.ffn = nn.Sequential(
            Linear(config.n_embed, dim_feedforward, bias=config.bias),
            nn.GELU(),
            Linear(dim_feedforward, config.n_embed, bias=config.bias),
            nn.Dropout(config.dropout)
        )
        self.lm_head = Linear(config.n_embed, config.vocab_size)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x: Tensor, xi: Tensor, targets: Tensor = None):
        if x.size(1) > self.pos_embed.size(1):
            x = x[:, :self.pos_embed.size(1)]
            if targets is not None:
                targets = targets[:, :self.pos_embed.size(1)]
        b, t = x.size()
        tok_embed = self.tok_embed(x)
        query = self.dropout(tok_embed + self.pos_embed[:, :t])
        mask = torch.triu(torch.ones((t, t), dtype=torch.bool, device=x.device), 1)
        res = query
        query = self.sa_ln(res)
        query, _ = self.sa_attn(query, query, query, attn_mask=mask)
        res = res + query
        query, _ = self.cross_attn(self.cross_ln(res), xi, xi)
        res = res + query
        res = res + self.ffn(self.ffn_ln(res))
        if targets is not None:
            logits = self.lm_head(res).float()
            loss = F.cross_entropy(logits.flatten(end_dim=1), targets.flatten(), ignore_index=183)
            return logits, loss
        return self.lm_head(res[:, [-1], :]).float(), None

class OCRModel(nn.Module):
    def __init__(self, config: ModelConfig = None):
        super().__init__()
        if config is None:
            config = ModelConfig()
        self.config = config
        self.tokenizer = tokenizer
        self.encoder = ImageEncoder(config)
        self.decoder = TextDecoder(config)

    def forward(self, img_tensor: Tensor, input_tokens: Tensor, targets_tokens: Tensor = None):
        xi = self.encoder(img_tensor)
        return self.decoder(input_tokens, xi, targets_tokens)

    @torch.inference_mode()
    def generate(self, img_tensor: Tensor, max_new_tokens: int = 128):
        if img_tensor.dim() == 3:
            img_tensor = img_tensor.unsqueeze(0)
        dev = next(self.parameters()).device
        img_tensor = img_tensor.to(dev)
        b = img_tensor.size(0)
        xi = self.encoder(img_tensor)
        idx = torch.full((b, 1), fill_value=self.tokenizer.bos_id, dtype=torch.long, device=dev)
        max_tokens = min(max_new_tokens, self.decoder.pos_embed.size(1) - 1)
        for _ in range(max_tokens):
            dec_out = self.decoder(idx, xi)
            logits = dec_out[0] if isinstance(dec_out, tuple) else dec_out
            logits = logits[:, -1, :]
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)
            idx = torch.cat((idx, idx_next), dim=1)
            if b == 1:
                if idx_next.item() == self.tokenizer.eos_id:
                    break
            else:
                if (idx == self.tokenizer.eos_id).any(dim=1).all():
                    break
        return idx

model = OCRModel(ModelConfig())
state_dict = torch.load(WEIGHTS_PATH, map_location="cpu")
if "state_dict" in state_dict:
    state_dict = state_dict["state_dict"]
model.load_state_dict(state_dict, strict=True)
print(f"✅ Pretrained Weights Loaded successfully from Hugging Face ({WEIGHTS_PATH})!")

## ជំហានទី ៩៖ ដាក់បញ្ចូល LoRA (Low-Rank Adaptation) លើ Transformer Blocks

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base_layer: nn.Linear, r: int = 16, lora_alpha: float = 32.0, lora_dropout: float = 0.05):
        super().__init__()
        self.base_layer = base_layer
        self.r = r
        self.lora_alpha = lora_alpha
        self.scaling = lora_alpha / r
        
        self.base_layer.weight.requires_grad = False
        if self.base_layer.bias is not None:
            self.base_layer.bias.requires_grad = False
            
        in_features = base_layer.in_features
        out_features = base_layer.out_features
        
        self.lora_A = nn.Parameter(torch.empty(r, in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features, r))
        self.dropout = nn.Dropout(p=lora_dropout) if lora_dropout > 0.0 else nn.Identity()
        
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base_out = self.base_layer(x)
        lora_out = (self.dropout(x) @ self.lora_A.t()) @ self.lora_B.t()
        return base_out + self.scaling * lora_out

    def merge_to_linear(self) -> nn.Linear:
        delta_w = (self.lora_B @ self.lora_A) * self.scaling
        self.base_layer.weight.data += delta_w.to(self.base_layer.weight.dtype)
        return self.base_layer

def apply_lora_to_model(ocr_model, r=16, lora_alpha=32.0, lora_dropout=0.05):
    for p in ocr_model.parameters():
        p.requires_grad = False
        
    for blk in ocr_model.encoder.blocks:
        blk.attn.qkv = LoRALinear(blk.attn.qkv, r=r, lora_alpha=lora_alpha, lora_dropout=lora_dropout)
        blk.attn.proj = LoRALinear(blk.attn.proj, r=r, lora_alpha=lora_alpha, lora_dropout=lora_dropout)
        blk.mlp.fc1 = LoRALinear(blk.mlp.fc1, r=r, lora_alpha=lora_alpha, lora_dropout=lora_dropout)
        blk.mlp.fc2 = LoRALinear(blk.mlp.fc2, r=r, lora_alpha=lora_alpha, lora_dropout=lora_dropout)
        
    ocr_model.decoder.ffn[0] = LoRALinear(ocr_model.decoder.ffn[0], r=r, lora_alpha=lora_alpha, lora_dropout=lora_dropout)
    ocr_model.decoder.ffn[2] = LoRALinear(ocr_model.decoder.ffn[2], r=r, lora_alpha=lora_alpha, lora_dropout=lora_dropout)
    
    trainable_params = sum(p.numel() for p in ocr_model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in ocr_model.parameters())
    print(f"🔥 LoRA Injected Successfully!")
    print(f"   - Trainable Parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
    print(f"   - Frozen Parameters:    {total_params - trainable_params:,}")
    print(f"   - Total Parameters:     {total_params:,}")
    return ocr_model

def merge_lora_model(ocr_model):
    for blk in ocr_model.encoder.blocks:
        if isinstance(blk.attn.qkv, LoRALinear): blk.attn.qkv = blk.attn.qkv.merge_to_linear()
        if isinstance(blk.attn.proj, LoRALinear): blk.attn.proj = blk.attn.proj.merge_to_linear()
        if isinstance(blk.mlp.fc1, LoRALinear): blk.mlp.fc1 = blk.mlp.fc1.merge_to_linear()
        if isinstance(blk.mlp.fc2, LoRALinear): blk.mlp.fc2 = blk.mlp.fc2.merge_to_linear()
    if isinstance(ocr_model.decoder.ffn[0], LoRALinear): ocr_model.decoder.ffn[0] = ocr_model.decoder.ffn[0].merge_to_linear()
    if isinstance(ocr_model.decoder.ffn[2], LoRALinear): ocr_model.decoder.ffn[2] = ocr_model.decoder.ffn[2].merge_to_linear()
    return ocr_model

model = apply_lora_to_model(model, r=16, lora_alpha=32.0, lora_dropout=0.05)
model.to(device)

## ជំហានទី ១០៖ កំណត់ Optimizer, Scheduler & ប្រព័ន្ធ Auto-Resume លើ Kaggle

⚙️ **ការកំណត់ការរក្សាទុក & បន្តការបង្វឹក (Auto-Resume):**
- ត្រួតពិនិត្យ `last_checkpoint.pt` ក្នុង `/kaggle/working/` បើមានវានឹង **Resume បន្តដោយស្វ័យប្រវត្តិតែម្ដង**
- `SAVE_EVERY_EPOCH = True`៖ រក្សាទុកឯកសារ LoRA Checkpoint **រៀងរាល់ចប់ 1 Epoch (Epoch 01, 02, ..., 30)** ក្នុង `epoch_checkpoints/`
- `SAVE_BEST = True`៖ រក្សាទុក **Best Model** (`best_khmer_parseq_lora_30epoch.pt`) នៅពេល CER ធ្លាក់ចុះទាបបំផុត
- `SAVE_LAST = True`៖ រក្សាទុក **Last State** (`last_checkpoint.pt`) សម្រាប់បន្តការងារ!

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler

EPOCHS = 30
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.01

SAVE_EVERY_EPOCH = True   # Save គ្រប់ Epoch ទាំងអស់ (Epoch 01 ដល់ 30)
SAVE_BEST = True          # Save Best Model (CER ទាបបំផុត)
SAVE_LAST = True          # Save Last Checkpoint សម្រាប់ Resume

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = GradScaler(enabled=torch.cuda.is_available())

best_model_path = os.path.join(OUTPUT_DIR, "best_khmer_parseq_lora_30epoch.pt")
last_model_path = os.path.join(OUTPUT_DIR, "last_checkpoint.pt")

# 🔄 ត្រួតពិនិត្យ និងដំណើរការ Auto-Resume ដោយស្វ័យប្រវត្តិលើ Kaggle
start_epoch = 1
best_val_cer = float("inf")
history = {"train_loss": [], "val_loss": [], "val_cer": []}

if os.path.exists(last_model_path):
    print(f"🔄 រកឃើញ Checkpoint ចាស់: {last_model_path}")
    print("⏳ កំពុងផ្ទុកទិន្នន័យដើម្បីបន្តការបង្វឹកដោយស្វ័យប្រវត្តិ (Auto-Resuming)...")
    try:
        saved_state = torch.load(last_model_path, map_location=device)
        model.load_state_dict(saved_state["model_state_dict"])
        optimizer.load_state_dict(saved_state["optimizer_state_dict"])
        scheduler.load_state_dict(saved_state["scheduler_state_dict"])
        start_epoch = saved_state["epoch"] + 1
        best_val_cer = saved_state.get("best_val_cer", float("inf"))
        history = saved_state.get("history", history)
        print(f"✅ បានភ្ជាប់ជោគជ័យ! បន្តបង្វឹកពី Epoch {start_epoch:02d} ដល់ {EPOCHS:02d} (Best CER បច្ចុប្បន្ន: {best_val_cer*100:.2f}%)")
    except Exception as e:
        print(f"⚠️ មិនអាច load checkpoint ចាស់បាន ({e}) -> ចាប់ផ្ដើមពី Epoch 01!")
        start_epoch = 1
else:
    print(f"🌱 មិនទាន់មាន Checkpoint ចាស់ -> ចាប់ផ្ដើមបង្វឹកពី Epoch 01 ដល់ {EPOCHS:02d}!")

print(f"✅ Optimizer: AdamW (lr={LEARNING_RATE})")
print(f"✅ Scheduler: CosineAnnealingLR (30 epochs)")
print(f"💾 Save Every Epoch: {SAVE_EVERY_EPOCH}")
print(f"💾 Save Best Model:  {SAVE_BEST}")

## ជំហានទី ១១៖ ដំណើរការបង្វឹក 30 Epochs (Training Loop លើ Kaggle ជាមួយ Auto-Resume)

In [ ]:
import time
import jiwer
from tqdm.auto import tqdm

if start_epoch > EPOCHS:
    print(f"🎉 ការបង្វឹកបានគ្រប់ {EPOCHS} Epochs រួចរាល់ហើយ!")
else:
    print("=" * 75)
    print(f"🏁 ចាប់ផ្ដើម Fine-Tuning LoRA (Epoch {start_epoch:02d} ដល់ {EPOCHS:02d}) លើ Kaggle GPU...")
    print("=" * 75)

for epoch in range(start_epoch, EPOCHS + 1):
    # --- 1. TRAINING PHASE ---
    model.train()
    running_loss = 0.0
    start_time = time.time()
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{EPOCHS:02d} [Train]", leave=False)
    for batch in pbar:
        imgs = batch["img"].to(device, non_blocking=True)
        inp_toks = batch["input_tokens"].to(device, non_blocking=True)
        tgt_toks = batch["target_tokens"].to(device, non_blocking=True)
        
        optimizer.zero_grad()
        with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
            logits, loss = model(imgs, inp_toks, tgt_toks)
            
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
        
    scheduler.step()
    avg_train_loss = running_loss / len(train_loader)
    epoch_time = time.time() - start_time
    
    # --- 2. VALIDATION PHASE ---
    model.eval()
    val_loss = 0.0
    val_batches = 0
    val_targets, val_predictions = [], []
    
    with torch.no_grad():
        for v_idx, batch in enumerate(val_loader):
            imgs = batch["img"].to(device)
            inp_toks = batch["input_tokens"].to(device)
            tgt_toks = batch["target_tokens"].to(device)
            
            with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
                _, loss = model(imgs, inp_toks, tgt_toks)
            val_loss += loss.item()
            val_batches += 1
            
            # Calculate CER safely using batch of 16
            if v_idx < 5:
                sub_imgs = imgs[:16]
                sub_tgts = tgt_toks[:16]
                pred_ids = model.generate(sub_imgs, max_new_tokens=64)
                for p_seq, t_seq in zip(pred_ids, sub_tgts):
                    p_text = tokenizer.decode(p_seq.tolist(), ignore_special_tokens=True)
                    t_text = tokenizer.decode(t_seq.tolist(), ignore_special_tokens=True)
                    val_predictions.append(p_text)
                    val_targets.append(t_text)
            
            if v_idx >= 30:
                break
                
    avg_val_loss = val_loss / max(val_batches, 1)
    current_cer = jiwer.cer(val_targets, val_predictions) if val_targets else 1.0
    
    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["val_cer"].append(current_cer)
    
    lora_state_dict = {k: v for k, v in model.state_dict().items() if "lora_" in k}
    
    # --- 3. SAVE CHECKPOINTS STRATEGY ---
    if SAVE_EVERY_EPOCH:
        epoch_ckpt_file = os.path.join(CHECKPOINTS_DIR, f"lora_epoch_{epoch:02d}_cer_{current_cer*100:.2f}.pt")
        torch.save({
            "epoch": epoch,
            "lora_state_dict": lora_state_dict,
            "train_loss": avg_train_loss,
            "val_loss": avg_val_loss,
            "val_cer": current_cer
        }, epoch_ckpt_file)
    
    is_best = current_cer < best_val_cer
    if is_best and SAVE_BEST:
        best_val_cer = current_cer
        torch.save(lora_state_dict, best_model_path)
        save_msg = f"⭐ NEW BEST! CER: {current_cer*100:.2f}% (Saved)"
    else:
        save_msg = ""
        
    if SAVE_LAST:
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_val_cer": best_val_cer,
            "history": history
        }, last_model_path)
        
    current_lr = scheduler.get_last_lr()[0]
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] ({epoch_time:.0f}s) | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val CER: {current_cer*100:.2f}% | LR: {current_lr:.2e} {save_msg}")

print("=" * 75)
print(f"🎉 ការបង្វឹក 30 Epochs បានបញ្ចប់ដោយជោគជ័យ! Best CER: {best_val_cer*100:.2f}%")
print(f"📁 All Epoch Checkpoints: {CHECKPOINTS_DIR}")
print(f"💾 Best Model:            {best_model_path}")
print(f"💾 Last Model (Resume):   {last_model_path}")

## ជំហានទី ១២៖ បញ្ចូលទម្ងន់ (Merge LoRA) ទៅជា Full Checkpoint Standalone លើ Kaggle

In [ ]:
if os.path.exists(best_model_path):
    lora_state = torch.load(best_model_path, map_location="cpu")
    model.load_state_dict(lora_state, strict=False)
    print(f"✅ Loaded best adapter weights from {best_model_path}")

model.cpu()
merged_model = merge_lora_model(model)

FINAL_CHECKPOINT = os.path.join(OUTPUT_DIR, "best_khmer_parseq_model_30epoch_finetuned.pt")
torch.save(merged_model.state_dict(), FINAL_CHECKPOINT)
print(f"🎉 SUCCESS! Standalone Merged Model saved to: {FINAL_CHECKPOINT}")
print(f"📁 File Size: {os.path.getsize(FINAL_CHECKPOINT) / 1e6:.2f} MB")

## ជំហានទី ១៣៖ គូសប្លង់ក្រាហ្វិក (Training Curves) នៃ Loss និង CER

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(range(1, len(history["train_loss"]) + 1), history["train_loss"], label="Train Loss", color="blue")
plt.plot(range(1, len(history["val_loss"]) + 1), history["val_loss"], label="Val Loss", color="red")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curves over 30 Epochs (Kaggle)")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(range(1, len(history["val_cer"]) + 1), [c * 100 for c in history["val_cer"]], label="Val CER (%)", color="green")
plt.xlabel("Epoch")
plt.ylabel("Character Error Rate (%)")
plt.title("Validation CER over 30 Epochs")
plt.legend()
plt.grid(True)

curve_path = os.path.join(OUTPUT_DIR, "training_curves.png")
plt.savefig(curve_path, bbox_inches="tight")
plt.show()
print(f"Graph saved to: {curve_path}")

## ជំហានទី ១៤៖ សាកល្បងតេស្ត Recognition លើរូបភាពគំរូនៃ Validation Set (Test Samples)

In [ ]:
merged_model.eval().to(device)

fig, axes = plt.subplots(3, 2, figsize=(14, 8))
axes = axes.flatten()

for i in range(min(6, len(val_raw))):
    sample = val_raw[i]
    img_pil = sample["image"]
    ground_truth = sample["text"]
    
    tensor = val_transform(img_pil.convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad():
        pred_ids = merged_model.generate(tensor, max_new_tokens=64)
        predicted = tokenizer.decode(pred_ids[0].tolist(), ignore_special_tokens=True)
        
    axes[i].imshow(img_pil)
    axes[i].set_title(f"GT:   {ground_truth}\nPred: {predicted}", fontsize=11, color="darkgreen" if ground_truth.strip() == predicted.strip() else "darkred")
    axes[i].axis("off")

plt.tight_layout()
plt.show()

## ជំហានទី ១៥៖ Zip ឯកសារទាំងអស់សម្រាប់ទាញយក (Download All Checkpoints in 1-Click)

នៅពេលបង្វឹកចប់ អ្នកគ្រាន់តែចូលទៅកាន់ផ្ទាំង **Output** នៃ Kaggle Notebook រួចចុច Download ឯកសារ `khmer_math_ocr_lora_results.zip` នោះអ្នកនឹងទទួលបានទាំង Checkpoint គ្រប់ Epochs និង Full Merged Model តែម្ដង!

In [ ]:
!cd /kaggle/working && zip -r khmer_math_ocr_lora_results.zip khmer_math_ocr_lora_30epoch

print("=" * 75)
print("📦 ឯកសារលទ្ធផលទាំងអស់ត្រូវបាន Zip រួចរាល់: /kaggle/working/khmer_math_ocr_lora_results.zip")
print("👉 អ្នកអាចទាញយក (Download) ឯកសារនេះបានដោយផ្ទាល់ពីផ្ទាំង Output នៃ Kaggle!")
print("=" * 75)